In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
IMG_C = 1

class CNetPlusScalar_simple(nn.Module):
    """
    CNetPlusScalar inherits from CNet but rewrites the forward method by introducing a scalar value to the output.
    The difference is that x is now a dictionary with the key 'image' and 'scalar'.
    old CNN applicable to 512x512 images
    """
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2) 
        self.conv1 = nn.Conv2d(IMG_C, 3, 5) 
        self.conv2 = nn.Conv2d(3, 3, 3) 
        self.conv3 = nn.Conv2d(3, 3, 3) 
        self.conv4 = nn.Conv2d(3, 3, 3)
        self.conv5 = nn.Conv2d(3, 3, 3)
        self.conv6 = nn.Conv2d(3, 3, 3)
        self.conv7 = nn.Conv2d(3, 3, 3)
        self.fc1 = nn.Linear(3 * 2 * 2, 30)
        self.fc2 = nn.Linear(30, 30)
        self.fc3 = nn.Linear(30+1, 1)

    def forward(self, image, vector):
        x = self.pool(F.relu(self.conv1(image))) 
        x = self.pool(F.relu(self.conv2(x))) 
        x = self.pool(F.relu(self.conv3(x))) 
        x = self.pool(F.relu(self.conv4(x))) 
        x = self.pool(F.relu(self.conv5(x)))
        x = self.pool(F.relu(self.conv6(x)))
        x = self.pool(F.relu(self.conv7(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.cat((x, vector), dim=1) # adding the scalar value
        x = self.fc3(x)
        return x

model = CNetPlusScalar_simple()

In [3]:
model.load_state_dict(torch.load("pre_trained_w.pt", weights_only=True))

RuntimeError: Error(s) in loading state_dict for CNetPlusScalar_simple:
	Missing key(s) in state_dict: "conv5.weight", "conv5.bias", "conv6.weight", "conv6.bias", "conv7.weight", "conv7.bias". 
	size mismatch for conv1.weight: copying a param with shape torch.Size([6, 1, 5, 5]) from checkpoint, the shape in current model is torch.Size([3, 1, 5, 5]).
	size mismatch for conv1.bias: copying a param with shape torch.Size([6]) from checkpoint, the shape in current model is torch.Size([3]).
	size mismatch for conv2.weight: copying a param with shape torch.Size([16, 6, 5, 5]) from checkpoint, the shape in current model is torch.Size([3, 3, 3, 3]).
	size mismatch for conv2.bias: copying a param with shape torch.Size([16]) from checkpoint, the shape in current model is torch.Size([3]).
	size mismatch for conv3.weight: copying a param with shape torch.Size([32, 16, 5, 5]) from checkpoint, the shape in current model is torch.Size([3, 3, 3, 3]).
	size mismatch for conv3.bias: copying a param with 

In [ ]:
tensor_x = torch.rand((1, 1, 512, 512), dtype=torch.float32)
scalar = torch.rand((1, 1), dtype=torch.float32)
torch.onnx.export(
    model,
    (tensor_x, scalar),
    f"{model.__class__.__name__}.onnx",  # filename of the ONNX model
    input_names=["input"],  # Rename inputs for the ONNX model
    dynamo=False,  # True or False to select the exporter to use
)

In [ ]:
!../onnx2c/build/onnx2c CNetPlusScalar_simple.onnx > C/CNetPlusScalar_simple.c